# Week 6 — Spark Architecture & Data Processing
**Celebal Technologies | Data Engineering Internship**  
**Name:** Harshita Gupta  
**Dataset:** Superstore Sales Data  

---
**Topics Covered:**
- Spark Architecture (Driver, Cluster Manager, Executor)
- Lazy Evaluation & DAG
- Reading CSV / Parquet with Schema
- Filtering, Selecting, Transforming DataFrames
- Wide Transformations, Shuffle, Predicate Pushdown
- Null Handling
- End-to-End Data Pipeline
- Best Practices for Large Datasets


## 🔧 Setup — Imports & SparkSession

In [1]:
!pip install pyspark -q
print("Done!")

Done!


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Week6").master("local[*]").getOrCreate()
print(f"✅ Spark {spark.version} ready!")

✅ Spark 4.0.3 ready!


In [3]:
import os, time, glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Week6_SparkAssignment")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"✅ SparkSession created")
print(f"   App Name : {spark.sparkContext.appName}")
print(f"   Master   : {spark.sparkContext.master}")
print(f"   Version  : {spark.version}")


✅ SparkSession created
   App Name : Week6
   Master   : local[*]
   Version  : 4.0.3


---
## Q1 — Roles of Driver, Cluster Manager, and Executor

| Component | Role |
|---|---|
| **Driver** | Entry point of the Spark app. Runs `main()`, creates `SparkSession`, builds the DAG, and schedules tasks via DAGScheduler. |
| **Cluster Manager** | Resource broker (Standalone / YARN / Kubernetes). Allocates CPU & memory to the app by assigning executor slots. |
| **Executor** | JVM process on each worker node. Executes the tasks sent by the Driver, stores RDD partitions in memory/disk, and reports results back. |

```
┌─────────────────────────────────────────┐
│           DRIVER PROGRAM                │  ← SparkSession lives here
│  builds DAG → schedules tasks           │
└─────────────────┬───────────────────────┘
                  │ submits tasks
         ┌────────▼────────┐
         │ CLUSTER MANAGER │  ← YARN / Standalone / K8s
         └────────┬────────┘
      ┌───────────┼───────────┐
      ▼           ▼           ▼
 ┌─────────┐ ┌─────────┐ ┌─────────┐
 │Executor │ │Executor │ │Executor │  ← run tasks in parallel
 └─────────┘ └─────────┘ └─────────┘
```


In [4]:
# Q1 — Demonstrating SparkSession (Driver) info
print("Driver Info:")
print(f"  App Name     : {spark.sparkContext.appName}")
print(f"  Master (Mode): {spark.sparkContext.master}")
print(f"  Default Cores: {spark.sparkContext.defaultParallelism}")
print()
print("In local[*] mode:")
print("  - Driver + Executor both run on THIS machine")
print("  - Cluster Manager is Spark's built-in local scheduler")


Driver Info:
  App Name     : Week6
  Master (Mode): local[*]
  Default Cores: 2

In local[*] mode:
  - Driver + Executor both run on THIS machine
  - Cluster Manager is Spark's built-in local scheduler


---
## Q2 — Lazy Evaluation & DAG

Spark does **NOT** execute transformations immediately.  
It records them as a logical plan (DAG) and only runs when an **action** is called.

**Flow:**
```
df.filter(...)         → NO execution  (adds to plan)
  .select(...)         → NO execution  (adds to plan)
  .withColumn(...)     → NO execution  (adds to plan)
  .count()   ← ACTION → NOW Spark executes the full DAG
```

**Why it matters:**  
- Catalyst Optimiser can reorder steps (e.g. push filter before join)  
- Eliminates redundant intermediate computations  
- Only data needed by the action is computed


In [8]:
# Q2 — Lazy Evaluation Demo
# Reading the CSV — NO execution yet, just a plan
df_lazy = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("superstore.csv")
)

# Adding transformations — still NO execution
df_lazy_filtered = df_lazy.filter(F.col("Category") == "Technology")
df_lazy_selected = df_lazy_filtered.select("Category", "Sales", "Profit")

print("Transformations added to plan — nothing executed yet!")
print("Calling count() now triggers the full DAG execution...")

# THIS is when Spark actually runs everything
count = df_lazy_selected.count()
print(f"\n✅ Action triggered! Technology rows: {count}")


Transformations added to plan — nothing executed yet!
Calling count() now triggers the full DAG execution...

✅ Action triggered! Technology rows: 1847


---
## Q3 — Read CSV with header + inferSchema


In [9]:
# Q3 — Reading CSV with inferSchema
df_infer = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("superstore.csv")
)

print("Schema (inferred):")
df_infer.printSchema()
print(f"Total Rows: {df_infer.count()}")
df_infer.show(5, truncate=True)


Schema (inferred):
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)

Total Rows: 9994
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-----------

**Best Practice — Explicit Schema (faster in production):**  
`inferSchema` does an extra full scan of the file to detect types.  
For large files, always define schema manually using `StructType`.


In [10]:
# Q3 — Reading CSV with EXPLICIT schema (best practice)
schema = StructType([
    StructField("Row_ID",        IntegerType(), True),
    StructField("Order_ID",      StringType(),  True),
    StructField("Order_Date",    StringType(),  True),
    StructField("Ship_Date",     StringType(),  True),
    StructField("Ship_Mode",     StringType(),  True),
    StructField("Customer_ID",   StringType(),  True),
    StructField("Customer_Name", StringType(),  True),
    StructField("Segment",       StringType(),  True),
    StructField("Country",       StringType(),  True),
    StructField("City",          StringType(),  True),
    StructField("State",         StringType(),  True),
    StructField("Postal_Code",   StringType(),  True),
    StructField("Region",        StringType(),  True),
    StructField("Product_ID",    StringType(),  True),
    StructField("Category",      StringType(),  True),
    StructField("Sub_Category",  StringType(),  True),
    StructField("Product_Name",  StringType(),  True),
    StructField("Sales",         DoubleType(),  True),
    StructField("Quantity",      IntegerType(), True),
    StructField("Discount",      DoubleType(),  True),
    StructField("Profit",        DoubleType(),  True),
])

df = (
    spark.read
    .option("header", "true")
    .schema(schema)
    .csv("superstore.csv")
)

print("Schema (explicit StructType):")
df.printSchema()
print(f"\nRows: {df.count()}  |  Columns: {len(df.columns)}")
df.show(5)


Schema (explicit StructType):
root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship_Date: string (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)


Rows: 9994  |  Columns: 21
+------+--------------+----------+----------+--------------+-----------+---------------

---
## Q4 — CSV vs Parquet: Storage & Performance

| Feature | CSV (row-based) | Parquet (columnar) |
|---|---|---|
| Layout | All columns stored per row | Each column stored separately |
| File size | Large (plain text) | Small (Snappy/ZSTD compression) |
| Schema | Not embedded | Embedded in file footer |
| Read speed | Slow — full file scan always | Fast — reads only needed columns |
| Predicate Pushdown | ❌ Not supported | ✅ Skips row-groups at file level |
| Best for | Data exchange, debugging | Analytics, production pipelines |

**Why it matters:**  
Selecting 3 columns from a 100-column CSV still reads ALL 100 columns.  
Parquet reads only those 3 column chunks → up to 97% less I/O.


In [11]:
# Q4 — Write to both formats and compare file sizes
os.makedirs("output", exist_ok=True)

# Write CSV
t0 = time.time()
df.write.mode("overwrite").option("header", "true").csv("output/superstore_csv")
csv_time = time.time() - t0

# Write Parquet
t0 = time.time()
df.write.mode("overwrite").parquet("output/superstore_parquet")
pq_time = time.time() - t0

# File size comparison
csv_size = sum(os.path.getsize(f) for f in glob.glob("output/superstore_csv/*.csv"))
pq_size  = sum(os.path.getsize(f) for f in glob.glob("output/superstore_parquet/*.parquet"))

print("=" * 45)
print(f"  CSV     size : {csv_size:>10,} bytes ({csv_size/1024:.1f} KB)")
print(f"  Parquet size : {pq_size:>10,} bytes ({pq_size/1024:.1f} KB)")
print(f"  Parquet is {csv_size/pq_size:.1f}x smaller ✅")
print("=" * 45)
print(f"  CSV write time    : {csv_time:.2f}s")
print(f"  Parquet write time: {pq_time:.2f}s")


  CSV     size :  2,287,006 bytes (2233.4 KB)
  Parquet size :    439,746 bytes (429.4 KB)
  Parquet is 5.2x smaller ✅
  CSV write time    : 1.51s
  Parquet write time: 3.31s


---
## Q5 — Select product_id and price where category = 'Electronics'

*(Using our dataset: `Product_ID`, `Sales` where `Category` = 'Technology')*


In [12]:
# Q5 — Filter + Select
result_q5 = (
    df
    .filter(F.col("Category") == "Technology")
    .select("Product_ID", "Sales", "Sub_Category")
)

print(f"Technology product rows: {result_q5.count()}")
result_q5.show(10)


Technology product rows: 1847
+---------------+--------+------------+
|     Product_ID|   Sales|Sub_Category|
+---------------+--------+------------+
|TEC-PH-10002275| 907.152|      Phones|
|TEC-PH-10002033| 911.424|      Phones|
|TEC-PH-10001949|  213.48|      Phones|
|TEC-AC-10003027|   90.57| Accessories|
|TEC-PH-10004977|1097.544|      Phones|
|TEC-PH-10000486| 371.168|      Phones|
|TEC-PH-10004093| 147.168|      Phones|
|TEC-AC-10000171|   45.98| Accessories|
|TEC-AC-10002167|    45.0| Accessories|
|TEC-PH-10003988|    21.8|      Phones|
+---------------+--------+------------+
only showing top 10 rows


---
## Q6 — Rename column and cast data type


In [13]:
# Q6 — Rename old_name → new_name, cast Sales (already Double, demo with Quantity)
df_revised = (
    df
    .withColumnRenamed("Sub_Category", "sub_category")   # rename
    .withColumnRenamed("Product_Name", "product_name")   # rename
    .withColumn("Sales", F.col("Sales").cast(DoubleType()))  # cast to Double
    .withColumn("Quantity", F.col("Quantity").cast(IntegerType()))
)

print("Revised Schema:")
df_revised.select("sub_category", "product_name", "Sales", "Quantity").printSchema()
df_revised.select("sub_category", "product_name", "Sales", "Quantity").show(5)


Revised Schema:
root
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)

+------------+--------------------+--------+--------+
|sub_category|        product_name|   Sales|Quantity|
+------------+--------------------+--------+--------+
|   Bookcases|Bush Somerset Col...|  261.96|       2|
|      Chairs|Hon Deluxe Fabric...|  731.94|       3|
|      Labels|Self-Adhesive Add...|   14.62|       2|
|      Tables|Bretford CR4500 S...|957.5775|       5|
|     Storage|Eldon Fold 'N Rol...|  22.368|       2|
+------------+--------------------+--------+--------+
only showing top 5 rows


---
## Q7 — Lineage Graph (DAG) and Fault Tolerance

Spark records every transformation as a **lineage graph (DAG)**.  
If a worker node fails:
1. Driver detects the lost partitions
2. Looks up the lineage — knows exactly which transformations produced them
3. **Re-computes only those lost partitions** from the last source/checkpoint
4. No full restart needed

This is more efficient than data replication because Spark re-runs a **deterministic computation** rather than maintaining redundant data copies.


In [14]:
# Q7 — View the lineage (logical plan) of a DataFrame
df_lineage = (
    df
    .filter(F.col("Region") == "West")
    .select("Order_ID", "Category", "Sales", "Profit")
    .withColumn("Margin", F.round(F.col("Profit") / F.col("Sales") * 100, 2))
)

print("Logical Plan (lineage) for df_lineage:")
print("=" * 55)
df_lineage.explain(mode="simple")


Logical Plan (lineage) for df_lineage:
== Physical Plan ==
*(1) Project [Order_ID#197, Category#210, Sales#213, Profit#216, round(((Profit#216 / Sales#213) * 100.0), 2) AS Margin#413]
+- *(1) Filter (isnotnull(Region#208) AND (Region#208 = West))
   +- FileScan csv [Order_ID#197,Region#208,Category#210,Sales#213,Profit#216] Batched: false, DataFilters: [isnotnull(Region#208), (Region#208 = West)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/superstore.csv], PartitionFilters: [], PushedFilters: [IsNotNull(Region), EqualTo(Region,West)], ReadSchema: struct<Order_ID:string,Region:string,Category:string,Sales:double,Profit:double>




---
## Q8 — Filter where status = 'Completed' AND amount > 1000

*(Using our dataset: Ship_Mode = 'First Class' AND Sales > 1000)*


In [15]:
# Q8 — Multiple filter conditions with AND
df_filtered_q8 = df.filter(
    (F.col("Ship_Mode") == "First Class") &
    (F.col("Sales") > 1000)
)

print(f"First Class orders with Sales > 1000: {df_filtered_q8.count()}")
df_filtered_q8.select("Order_ID", "Ship_Mode", "Sales", "Profit").show(10)


First Class orders with Sales > 1000: 71
+--------------+-----------+--------+---------+
|      Order_ID|  Ship_Mode|   Sales|   Profit|
+--------------+-----------+--------+---------+
|CA-2016-117590|First Class|1097.544| 123.4737|
|CA-2016-129714|First Class|4355.168|1415.4296|
|CA-2014-154627|First Class|2735.952|  341.994|
|CA-2016-110499|First Class|1199.976| 374.9925|
|CA-2014-151995|First Class| 1298.55|  311.652|
|CA-2016-158099|First Class| 1141.47|  -760.98|
|CA-2017-159366|First Class|3059.982|  679.996|
|CA-2014-136567|First Class| 2244.48| 493.7856|
|CA-2016-133711|First Class|  3040.0|   1459.2|
|CA-2016-152289|First Class|1024.716| -29.2776|
+--------------+-----------+--------+---------+
only showing top 10 rows


---
## Q9 — Predicate Pushdown in Parquet

Parquet stores **column statistics** (min, max, null count) per row-group in its file footer.

When Spark reads Parquet with a `filter()`:
- Catalyst Optimiser **pushes the filter down** to the Parquet reader
- Reader checks statistics and **skips entire row-groups** that can't match
- Those blocks are **never loaded into memory**
- Result → far less I/O, fewer tasks, much faster queries

Verify pushdown is active using `.explain("formatted")` — look for `PushedFilters`.


In [16]:
# Q9 — Predicate Pushdown demo on Parquet
df_parquet = spark.read.parquet("output/superstore_parquet")

df_pushdown = df_parquet.filter(F.col("Category") == "Technology")

print("Physical Plan — look for 'PushedFilters' below:")
print("=" * 55)
df_pushdown.explain(mode="formatted")


Physical Plan — look for 'PushedFilters' below:
== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [21]: [Row_ID#458, Order_ID#459, Order_Date#460, Ship_Date#461, Ship_Mode#462, Customer_ID#463, Customer_Name#464, Segment#465, Country#466, City#467, State#468, Postal_Code#469, Region#470, Product_ID#471, Category#472, Sub_Category#473, Product_Name#474, Sales#475, Quantity#476, Discount#477, Profit#478]
Batched: true
Location: InMemoryFileIndex [file:/content/output/superstore_parquet]
PushedFilters: [IsNotNull(Category), EqualTo(Category,Technology)]
ReadSchema: struct<Row_ID:int,Order_ID:string,Order_Date:string,Ship_Date:string,Ship_Mode:string,Customer_ID:string,Customer_Name:string,Segment:string,Country:string,City:string,State:string,Postal_Code:string,Region:string,Product_ID:string,Category:string,Sub_Category:string,Product_Name:string,Sales:double,Quantity:int,Discount:double,Profit:double>

(2) ColumnarToRow [codegen i

---
## Q10 — Add new column final_price = base_price × 1.18 (18% tax)

*(Using Sales as base_price)*


In [17]:
# Q10 — Add derived column with 18% tax
df_tax = df.withColumn(
    "final_price",
    F.round(F.col("Sales") * 1.18, 2)
)

df_tax.select("Order_ID", "Sales", "final_price").show(10)
print(f"\nfinal_price = Sales × 1.18  ✅")


+--------------+--------+-----------+
|      Order_ID|   Sales|final_price|
+--------------+--------+-----------+
|CA-2016-152156|  261.96|     309.11|
|CA-2016-152156|  731.94|     863.69|
|CA-2016-138688|   14.62|      17.25|
|US-2015-108966|957.5775|    1129.94|
|US-2015-108966|  22.368|      26.39|
|CA-2014-115812|   48.86|      57.65|
|CA-2014-115812|    7.28|       8.59|
|CA-2014-115812| 907.152|    1070.44|
|CA-2014-115812|  18.504|      21.83|
|CA-2014-115812|   114.9|     135.58|
+--------------+--------+-----------+
only showing top 10 rows

final_price = Sales × 1.18  ✅


---
## Q11 — Transformations vs Actions

| Type | Behaviour | Examples |
|---|---|---|
| **Transformation** | Lazy — builds the DAG, nothing runs | `filter()`, `select()`, `withColumn()`, `groupBy()`, `join()` |
| **Action** | Eager — triggers DAG execution | `count()`, `show()`, `collect()`, `write()`, `first()` |

⚠️ `collect()` pulls ALL rows to Driver memory → **never use on large data!**


In [18]:
# Q11 — Transformations (lazy) vs Actions (trigger execution)

# TRANSFORMATIONS — nothing runs yet
t1 = df.filter(F.col("Profit") > 0)                        # transformation 1
t2 = t1.select("Category", "Region", "Sales", "Profit")    # transformation 2
t3 = t2.withColumn("Margin", F.round(F.col("Profit") / F.col("Sales") * 100, 2))  # transformation 3

print("3 transformations added — zero execution so far!")
print()

# ACTIONS — these trigger execution
print("Action 1 — count():")
print(f"  Profitable rows: {t3.count()}")

print("\nAction 2 — show():")
t3.show(5)

print("\nAction 3 — first():")
print(f"  First row: {t3.first()}")


3 transformations added — zero execution so far!

Action 1 — count():
  Profitable rows: 7994

Action 2 — show():
+---------------+------+------+-------+------+
|       Category|Region| Sales| Profit|Margin|
+---------------+------+------+-------+------+
|      Furniture| South|261.96|41.9136|  16.0|
|      Furniture| South|731.94|219.582|  30.0|
|Office Supplies|  West| 14.62| 6.8714|  47.0|
|Office Supplies| South|22.368| 2.5164| 11.25|
|      Furniture|  West| 48.86|14.1694|  29.0|
+---------------+------+------+-------+------+
only showing top 5 rows

Action 3 — first():
  First row: Row(Category='Furniture', Region='South', Sales=261.96, Profit=41.9136, Margin=16.0)


---
## Q12 — Load Parquet → filter null user_id → save as CSV


In [20]:
# Q12 — Full pipeline: Parquet in → filter nulls → CSV out
df_parquet_in = spark.read.parquet("output/superstore_parquet")

print(f"Rows before null filter: {df_parquet_in.count()}")

# Filter out rows where Customer_ID is null
df_clean_q12 = df_parquet_in.filter(F.col("Customer_ID").isNotNull())

print(f"Rows after null filter : {df_clean_q12.count()}")

# Save as CSV
df_clean_q12.write.mode("overwrite").option("header", "true").csv("output/q12_output_csv")

print("\n✅ Saved to output/q12_output_csv/")


Rows before null filter: 9994
Rows after null filter : 9994

✅ Saved to output/q12_output_csv/


---
## Q13 — Client Mode vs Cluster Mode

| Aspect | Client Mode | Cluster Mode |
|---|---|---|
| **Driver location** | Runs on the submitting machine (your laptop/edge node) | Runs inside the cluster on a worker node |
| **Logs** | Printed directly to your terminal | Stored in the cluster — use Spark UI |
| **Network** | Driver ↔ Executors: cross-network (slow if disconnected) | Driver & Executors co-located → low latency |
| **Best for** | Development, Jupyter notebooks, debugging | Production pipelines, automated jobs |

In **local mode** (what we use here), both Driver and Executor run on the same machine — it's essentially Client mode for single-machine development.


In [21]:
# Q13 — Show current execution mode
print(f"Current Master : {spark.sparkContext.master}")
print()
print("local[*]  → local mode (development)")
print("yarn      → YARN cluster (production on Hadoop)")
print("k8s://... → Kubernetes cluster (containerised)")
print()
print("In production you would submit like:")
print("  spark-submit --master yarn --deploy-mode cluster script.py   (Cluster Mode)")
print("  spark-submit --master yarn --deploy-mode client script.py    (Client Mode)")


Current Master : local[*]

local[*]  → local mode (development)
yarn      → YARN cluster (production on Hadoop)
k8s://... → Kubernetes cluster (containerised)

In production you would submit like:
  spark-submit --master yarn --deploy-mode cluster script.py   (Cluster Mode)
  spark-submit --master yarn --deploy-mode client script.py    (Client Mode)


---
## Q14 — Filter where region = 'North' OR priority = 'High'

*(Using our dataset: Region = 'West' OR Segment = 'Corporate')*


In [23]:
# Q14 — OR condition filter
df_filtered_q14 = df.filter(
    (F.col("Region") == "West") |
    (F.col("Segment") == "Corporate")
)

print(f"West OR Corporate rows: {df_filtered_q14.count()}")
df_filtered_q14.select("Order_ID", "Region", "Segment", "Sales").show(10)


West OR Corporate rows: 5263
+--------------+------+---------+--------+
|      Order_ID|Region|  Segment|   Sales|
+--------------+------+---------+--------+
|CA-2016-138688|  West|Corporate|   14.62|
|CA-2014-115812|  West| Consumer|   48.86|
|CA-2014-115812|  West| Consumer|    7.28|
|CA-2014-115812|  West| Consumer| 907.152|
|CA-2014-115812|  West| Consumer|  18.504|
|CA-2014-115812|  West| Consumer|   114.9|
|CA-2014-115812|  West| Consumer|1706.184|
|CA-2014-115812|  West| Consumer| 911.424|
|CA-2016-161389|  West| Consumer| 407.976|
|CA-2014-167164|  West| Consumer|    55.5|
+--------------+------+---------+--------+
only showing top 10 rows


---
## Q15 — Why .show(5) is safer than .collect() on large data


In [24]:
# Q15 — show() vs collect() demo

# ✅ SAFE — show(5) fetches only 5 rows to Driver
print("show(5) — safe on any size dataset:")
df.select("Order_ID", "Category", "Sales").show(5)

# ⚠️ UNSAFE on large data — collect() pulls ALL rows to Driver RAM
# On a multi-TB dataset this causes OutOfMemoryError and crashes the Driver
# Here we limit first so it's safe for demo
print("collect() on small limit (safe for demo only):")
rows = df.select("Order_ID").limit(3).collect()
print(f"  Collected rows: {[r['Order_ID'] for r in rows]}")

print()
print("=" * 55)
print("  RULE: Never use collect() on large/production data!")
print("  USE : show(n) to inspect, write() to save results")
print("=" * 55)


show(5) — safe on any size dataset:
+--------------+---------------+--------+
|      Order_ID|       Category|   Sales|
+--------------+---------------+--------+
|CA-2016-152156|      Furniture|  261.96|
|CA-2016-152156|      Furniture|  731.94|
|CA-2016-138688|Office Supplies|   14.62|
|US-2015-108966|      Furniture|957.5775|
|US-2015-108966|Office Supplies|  22.368|
+--------------+---------------+--------+
only showing top 5 rows
collect() on small limit (safe for demo only):
  Collected rows: ['CA-2016-152156', 'CA-2016-152156', 'CA-2016-138688']

  RULE: Never use collect() on large/production data!
  USE : show(n) to inspect, write() to save results


---
## 🚀 Bonus — End-to-End Data Pipeline

**Flow:** Read CSV → Clean Nulls → Enrich → Filter → Aggregate → Write Parquet


In [25]:
# BONUS — Full pipeline

# Step 1: Read
df_raw = spark.read.option("header","true").schema(schema).csv("superstore.csv")
print(f"[1] Raw rows: {df_raw.count()}")

# Step 2: Clean
df_clean = (df_raw
    .fillna({"Profit": 0.0, "Discount": 0.0, "Sales": 0.0})
    .dropDuplicates(["Order_ID", "Product_ID"])
    .withColumn("Order_Date", F.to_date("Order_Date", "M/d/yyyy"))
)
print(f"[2] Clean rows: {df_clean.count()}")

# Step 3: Enrich
df_enriched = (df_clean
    .withColumn("Revenue",      F.round(F.col("Sales") * F.col("Quantity"), 2))
    .withColumn("Profit_Margin",F.round(F.col("Profit") / F.when(F.col("Sales")!=0, F.col("Sales")).otherwise(1) * 100, 2))
    .withColumn("Is_Profitable",F.when(F.col("Profit") > 0, "Yes").otherwise("No"))
    .withColumn("Sales_Tier",   F.when(F.col("Sales") >= 1000, "High")
                                 .when(F.col("Sales") >= 300, "Medium")
                                 .otherwise("Low"))
)
print(f"[3] Enriched with 4 new columns")

# Step 4: Filter
df_final = df_enriched.filter(
    (F.col("Is_Profitable") == "Yes") & (F.col("Sales") > 0)
)
print(f"[4] Profitable rows: {df_final.count()}")

# Step 5: Aggregate
print("\n[5] Category Summary:")
df_final.groupBy("Category").agg(
    F.count("*").alias("orders"),
    F.round(F.sum("Revenue"), 2).alias("total_revenue"),
    F.round(F.avg("Profit_Margin"), 2).alias("avg_margin_pct")
).orderBy("total_revenue", ascending=False).show()

# Step 6: Write partitioned Parquet
df_final.write.mode("overwrite").partitionBy("Category").parquet("output/final_pipeline")
print("[6] ✅ Written to output/final_pipeline/ (partitioned by Category)")


[1] Raw rows: 9994
[2] Clean rows: 9986
[3] Enriched with 4 new columns
[4] Profitable rows: 7819

[5] Category Summary:
+---------------+------+-------------+--------------+
|       Category|orders|total_revenue|avg_margin_pct|
+---------------+------+-------------+--------------+
|     Technology|  1563|   3493608.36|         22.58|
|Office Supplies|  4914|   2991854.48|         33.12|
|      Furniture|  1342|   2455646.87|         22.76|
+---------------+------+-------------+--------------+

[6] ✅ Written to output/final_pipeline/ (partitioned by Category)


---
## ✅ Assignment Complete!

| # | Topic | Status |
|---|---|---|
| Q1 | Driver, Cluster Manager, Executor | ✅ |
| Q2 | Lazy Evaluation & DAG | ✅ |
| Q3 | Read CSV with header + inferSchema | ✅ |
| Q4 | CSV vs Parquet | ✅ |
| Q5 | Filter + Select | ✅ |
| Q6 | Rename + Cast | ✅ |
| Q7 | Lineage Graph & Fault Tolerance | ✅ |
| Q8 | Multi-condition Filter (AND) | ✅ |
| Q9 | Predicate Pushdown in Parquet | ✅ |
| Q10 | Add derived column (tax) | ✅ |
| Q11 | Transformations vs Actions | ✅ |
| Q12 | Parquet → filter nulls → CSV | ✅ |
| Q13 | Client Mode vs Cluster Mode | ✅ |
| Q14 | Multi-condition Filter (OR) | ✅ |
| Q15 | show() vs collect() | ✅ |
| Bonus | End-to-End Pipeline | ✅ |


In [26]:
# Stop SparkSession
spark.stop()
print("SparkSession stopped. Assignment complete! ✅")


SparkSession stopped. Assignment complete! ✅
